<a href="https://colab.research.google.com/github/justusjb/ledits_pp/blob/main/kayak_shenanigans_oneway.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Säkerhetskoll

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time


def get_response_for_url(url):
  payload = {}
  headers = {
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'accept-language': 'en-US,en;q=0.9',
    'priority': 'u=0, i',
    'sec-ch-ua': '"Brave";v="125", "Chromium";v="125", "Not.A/Brand";v="24"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
    'sec-fetch-dest': 'document',
    'sec-fetch-mode': 'navigate',
    'sec-fetch-site': 'none',
    'sec-fetch-user': '?1',
    'sec-gpc': '1',
    'upgrade-insecure-requests': '1',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
  }

  response = requests.request("GET", url, headers=headers, data=payload)
  return response




In [ ]:

def get_searchid_for_response(response):
  soup = BeautifulSoup(response.text, 'html.parser')


  # Find the <script> tag with the specific id
  script_tag = soup.find('script', id='__R9_HYDRATE_DATA__')

  # Extract the content of the <script> tag
  if script_tag:
      json_content = script_tag.string.strip()

      # Parse the JSON content
      data = json.loads(json_content)

      #print("DATA: JSON CONTENT")
      #print(data)


      def find_search_id(obj):
          if isinstance(obj, dict):
              for key, value in obj.items():
                  if key == 'searchId':
                      return value
                  result = find_search_id(value)
                  if result is not None:
                      return result
          elif isinstance(obj, list):
              for item in obj:
                  result = find_search_id(item)
                  if result is not None:
                      return result
          return None


      # Find the searchID
      search_id = find_search_id(data)
      if search_id:
          return search_id
      else:
          raise Exception("searchID not found")
  else:
      raise Exception("Script tag not found")

In [ ]:

def nth_poll(search_id, from_to, url, n):
  url = f"https://www.kayak.de/s/horizon/flights/results/FlightSearchPollAction?p={n}"

  payload = f'priceType=daybase&searchId={search_id}&url=%2Fflights%2F{from_to}%2F2024-10-04-flexible-3days%2F1students&pageNumber=1&pollNumber=2&requestReason=POLL&isSecondPhase=false&sortMode=price&ascending=true'

  headers = {
  'accept': '*/*',
  'accept-language': 'en-US,en;q=0.9',
  'content-type': 'application/x-www-form-urlencoded',
  'origin': 'https://www.kayak.de',
  'priority': 'u=1, i',
  'referer': url,
  'sec-ch-ua': '"Brave";v="125", "Chromium";v="125", "Not.A/Brand";v="24"',
  'sec-ch-ua-mobile': '?0',
  'sec-ch-ua-platform': '"macOS"',
  'sec-fetch-dest': 'empty',
  'sec-fetch-mode': 'cors',
  'sec-fetch-site': 'same-origin',
  'sec-gpc': '1',
  'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
  'x-csrf': 'HW0$K2ff2FdAQcZAududKYed02SQaG9tF9RqSBeRfq0-olnMSXY_SIZPJ7YZT$fwa$837hMQB48EdcX4jRUdjA8',
  'x-requested-with': 'XMLHttpRequest'
  }

  for attempt in range(3):
    try:
      response = requests.request("POST", url, data=payload, headers=headers)
      break
    except requests.exceptions.ChunkedEncodingError:
      time.sleep(1)
  else:
    raise Exception("Chunking error")

  return response.text


In [ ]:
#from_to = "BER-SFO"

def get_flight_price(from_to):
  url = f"https://www.kayak.de/flights/{from_to}/2024-10-04-flexible-3days/1students?sort=price_a"

  resp = get_response_for_url(url)
  searchid = get_searchid_for_response(resp)
  #print("SEARCHID", searchid)
  poll = nth_poll(searchid, from_to, url, 0)
  #print(poll)
  poll_status = json.loads(poll)["PollStatus"]
  if not poll_status['error'] and not poll_status['errors']:
    # keep polling
    search_completed = poll_status['searchCompleted']
    search_number = 1
    while not search_completed:
      poll = nth_poll(searchid, from_to, url, search_number)
      #print(poll)
      poll_status = json.loads(poll)["PollStatus"]
      #print(poll_status['error'])
      #print(poll_status['errors'])
      #print(poll_status['searchCompleted'])
      if poll_status['error'] and poll_status['errors']:
        raise Exception(f"Polling error in Poll {search_number}:", poll_status['error'], poll_status['errors'])
      search_completed = poll_status['searchCompleted']
      search_number += 1
      #print()
      time.sleep(1)
  else:
    raise Exception(f"Polling errorin Poll 0:", poll_status['error'], poll_status['errors'])

  poll_json = json.loads(poll)

  results = poll_json['FlightResultsList']['results']
  #print(results)
  for res in results.values():
    if not 'resultId' in res:
      continue
    else:
      #print(res['isCheapest'])
      return (res['trackingDataLayer']['tagLayerPrice'], res['optionsByFare'][0]['options'][0]['displayPrice'])
      break


metropolis = ["LON,PAR,AMS", "MAD,BCN,VNO", "PMI,GDN,POZ", "ATH,MAN,AGP", "WAW,HEL,BRU", "LUX"]
germany = ["AGB,BER,BWE", "BRE,CGN,DTM", "DRS,DUS,ERF", "FRA,FDH,HHN", "HAM,HAJ,HDF", "FKB,KSF,LEJ", "LBC,MHG,FMM", "MUC,FMO,NUE", "PAD,RLG,SCN", "STR,NRN,GWT"]
austria = ["GRZ,INN,KLU", "LNZ,SZG,VIE"]
swiss = ["EAP,BRN,GVA", "LUG,ACH,ZRH"]
nordics_sas = ["AAL,AAR,BLL", "CPH,AGH,OSD", "GOT,KRN,LLA", "MMX,RNB,SCR", "SFT,STO,SDL", "UME,VBY,AES", "ALF,BGO,BOO", "EVE,HAU,KKN", "KRS,KSU,LYR", "OSL,SVG,TOS", "TRD"]
italy_sas = ["BRI,BLQ,CTA", "FLR,GOA,MIL", "NAP,OLB,PMO", "PSA,ROM,TRN", "VCE"]
all_airports = metropolis + germany + austria + swiss + nordics_sas + italy_sas
for airports in all_airports:
  try:
    print(airports, get_flight_price(airports + "-KTM"))
  except Exception as e:
    print(airports, e.message)


LON,PAR,AMS (306, '306\xa0€')
MAD,BCN,VNO (288, '288\xa0€')
PMI,GDN,POZ (304, '304\xa0€')
ATH,MAN,AGP (275, '275\xa0€')
WAW,HEL,BRU (303, '303\xa0€')
LUX (359, '359\xa0€')
AGB,BER,BWE (302, '302\xa0€')
BRE,CGN,DTM (339, '339\xa0€')
DRS,DUS,ERF (342, '342\xa0€')
FRA,FDH,HHN (358, '358\xa0€')
HAM,HAJ,HDF (312, '312\xa0€')
FKB,KSF,LEJ (355, '355\xa0€')
LBC,MHG,FMM (306, '306\xa0€')
MUC,FMO,NUE (317, '317\xa0€')
PAD,RLG,SCN (436, '436\xa0€')
STR,NRN,GWT (304, '304\xa0€')
GRZ,INN,KLU (421, '421\xa0€')
LNZ,SZG,VIE (293, '293\xa0€')
EAP,BRN,GVA (308, '308\xa0€')
LUG,ACH,ZRH (309, '309\xa0€')
AAL,AAR,BLL (313, '313\xa0€')
CPH,AGH,OSD (337, '337\xa0€')
GOT,KRN,LLA (340, '340\xa0€')
MMX,RNB,SCR (335, '335\xa0€')
UME,VBY,AES (410, '410\xa0€')
ALF,BGO,BOO (383, '383\xa0€')
EVE,HAU,KKN (405, '405\xa0€')
KRS,KSU,LYR (402, '402\xa0€')
OSL,SVG,TOS (340, '340\xa0€')
TRD (372, '372\xa0€')
BRI,BLQ,CTA (282, '282\xa0€')
FLR,GOA,MIL (283, '283\xa0€')
NAP,OLB,PMO (296, '296\xa0€')
PSA,ROM,TRN (268, '268\xa0

In [ ]:

poll_json = json.loads(poll)
poll_status = poll_json["PollStatus"]
print(poll_status['error'])
print(poll_status['errors'])
print(poll_status['searchCompleted'])

results = poll_json['FlightResultsList']['results']
print(results)
for res in results.values():
  if not 'resultId' in res:
    continue
  else:
    print(res['isCheapest'])
    print(res['trackingDataLayer']['tagLayerPrice'])
    print(res['optionsByFare'][0]['options'][0]['displayPrice'])
    break

None
None
True
{'inline-1-1': {'clickUrl': '/s/clickthrough.jsp?ctyp=InlineOpaqueAdBooking&ptyp=F&orig=F..RP..M1&octid=&plid=5284207&cpnid=7094905&pid=CondorDach_FIOAD_de_DE&prv=CondorDach_FIOAD_de_DE&srch=lBBiKLbB0u&ploc=de_DE&displayRail=Rslt&rank=1&atype=inline&prc=1121.57&pgrp=0&xpExt=&aidExt=&lid=CondorDach_FIOAD_de_DE-lBBiKLbB0u&qorig=Airport:MUC&qdest=Airport:SFO&qstart=1718769600000&qend=1719460800000&qtravelers=1&qrooms=0&qow=false&qfcc=e&qdctid=13852&qdac=SFO&qshour=-1&qehour=-1&resid=1b4eaf23521b6b6864e9f969f4060937&bookid=F-2703736927212002661E0c771208f9c&seekProv=DE&qadults=1&qchild=0&qns=false&qnearby=0&qnearbyo=false&qnearbyd=false&qcages=&qinfantseat=0&qinfantlap=0&qsenior=0&pgNum=1&copyId=0&iar=1&stid=172087351&ah=53u9Q6Og48hG3fU0x8BbDE0SLrbntAthNedgciH8Qbg&pj=ekPUspihZt375AkUbW3nag%3D%3D&_sid_=R-43veCO_tuylSVqV7Bf_9t-__toegoR7jcBoEkOVAveCY5PfApM6LRtEc1ESygJR&prvcurl=&btype=DgqbZHoRrmSpOXmNAG7plg%3D%3D&bkey=7q5Mu%24vEmzlff5KBZYDJew%3D%3D&abv=JwBN07Jre4E%3D', 'price': '

In [ ]:
for x in range(3):
  print(x)
  if x == 2:
    break
else:
  print("LLL")

0
1
2
